### Lab 4: Working with CNN models and Transfer Learning using Keras API

Estimated time: **40-45 minutes**

Accelerator: **T4 GPU**

In this lab we learn how custom deep learning models are trained using Keras APIs.
*   Train & evaluate LeNet based and transfer learning based CNN model for sound classification

##### **Step 1:**  

- Change to a GPU based compute server


##### **Step 2:**

Develop a LeNet style CNN Model for 11 Environment Sound Category (ESC-11).  At a high level, LeNet (LeNet-5) consists of two parts:
- A convolutional encoder consisting of two convolutional layers;
- And a dense block consisting of three fully-connected layers.  

##### **Step 2(a):**

Download ESC-11 and organize the sound dataset (it contains 11 sound categories).


In [0]:
!wget -q https://edge-ai-doulos.s3.us-west-2.amazonaws.com/esc_spec.zip
!unzip -q esc_spec

##### **Step 2(b):**

Use [`ImageDataGenerator`](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image/ImageDataGenerator) to rescale the image data into float values (divide by 255 so the tensor values are between 0 and 1), and call `flow_from_directory()` to create two generators: one for the training dataset and one for the validation dataset.

In [0]:
import os
import tensorflow as tf
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  ## hide INFO/WARNING logs


from tensorflow.keras.preprocessing.image import ImageDataGenerator

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'esc_spec')
LABEL_FILE = os.path.join(BASE_DIR, 'sound_labels.txt')
MODEL_SAVE_PATH = os.path.join(BASE_DIR, 'ESC-11-LENET-CNN.tflite')

IMAGE_SIZE = 224; BATCH_SIZE = 5

datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.3)

train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size= BATCH_SIZE,
    subset='training')

val_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size= BATCH_SIZE,
    subset='validation')

#####  **Exercise**: 
Use of new API to replace datagen.flow_from_directory
- There is a new API **tf.keras.preprocessing.image_dataset_from_directory** to replace datagen.flow_from_directory.
- As an exercise, you can update this notebook to use the new API in all the cells where datagen.flow_from_directory is used.
- Learn to use this new API from the Keras documentation for future projects.


##### Solution

<details>
    <summary> Click here to see use of new API </summary>

    import os
    import tensorflow as tf

    # Each subdirectory contains images belonging to that class.
    BASE_DIR = os.getcwd()
    DATA_DIR = os.path.join(BASE_DIR, 'esc_spec')
    LABEL_FILE = os.path.join(BASE_DIR, 'sound_labels.txt')

    # Create a training dataset
    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        labels='inferred',
        label_mode='int',
        image_size=(224, 224),
        batch_size=5,
        shuffle=True,
        seed=42,
        validation_split=0.2,
        subset='training'
)

    # Create a validation dataset
    val_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        labels='inferred',
        label_mode='int',
        image_size=(128, 128),
        batch_size=32,
        shuffle=True,
        seed=42,
        validation_split=0.2,
        subset='validation'
    )

    # Access class names
    class_names = train_ds.class_names
    print(f"Class names: {class_names}")

    # Iterate through the dataset
    for images, labels in train_ds.take(1):
        print(f"Batch of images shape: {images.shape}")
        print(f"Batch of labels shape: {labels.shape}")
   
</details>

##### **Step 2(c):**

On each iteration, these generators provide a batch of images by reading images from disk and processing them to the proper tensor size (224 x 224). The output is a tuple of (images, labels). For example, you can see the shapes here:

In [0]:
image_batch, label_batch = next(val_generator)
image_batch.shape, label_batch.shape

In [0]:
image_batch, label_batch = next(train_generator)
image_batch.shape, label_batch.shape

##### **Step 2(d):**

Next we save the class labels to a text file called (sound_labels.txt):
The generated txt file is explored using cat.

In [0]:
print (train_generator.class_indices)

labels = '\n'.join(sorted(train_generator.class_indices.keys()))

with open(LABEL_FILE, 'w') as f:
  f.write(labels)

In [0]:
!cat sound_labels.txt

##### **Step 2(e):**

Create LeNet style CNN model using four sets of 2D convolution and max pooling layers for feature extraction and two dense layers for classication.  


In [0]:
import keras
from keras import layers, Input
from keras import models

model = models.Sequential()

model.add(keras.Input(shape=(224,224,3)))
# Feature Detection Layers
model.add(layers.Conv2D(32, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())

#Classification layers
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dropout(0.5)),
model.add(layers.Dense(11, activation='softmax'))

##### **Step 2(f):**

Compilation step: This step configures the model's optimizer, loss and metric. Since we have multiple (11) categories sound to classify, we use categorical_crossentropy as loss. The model.summary() is used to reconfirm model's architecture.

In [0]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

##### Question: 
Why is categorial crossentropy used as loss function instead of binary crossentropy
  

##### Answer

<details> 
    <summary> Click here for our answer </summary>

    Binary crossentropy is used as the loss function when we have two categories (such as dogs/cats). For our model we have 11 categories, as a result we use categorical crossentropy.

</details>



##### **Step 2(g): Training LeNet model on spectrograms**

Now we can train the model using data provided by the train_generator and val_generator that we created at the beginning. *Watch the GPU utilization as the model trains in the terminal window above*.


In [0]:
history = model.fit(train_generator,
                    steps_per_epoch=len(train_generator),
                    epochs=25,
                    validation_data=val_generator,
                    validation_steps=len(val_generator))

##### **Step 2(h):**

Review the learning curves of LeNet based CNN Model on ESC-11


In [0]:
import matplotlib.pyplot as plt


acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
#plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),1])
plt.title('Validation Accuracy')
plt.show()

##### **Experiment:**

- Uncomment line 11 to plot the training accuracy.
- What does the gap between training and validation accuracy mean?
- Do you think a lower gap in training and validation accuracy characterizes a robust model?

##### **Step 2(i):**

Take a look at the validation accuracy curve of the LeNet CNN model, after 20 epochs of training on the ESC-11 dataset.  Save the model in TensorFlow Lite format for later use.

In [0]:
# Convert the model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with tf.io.gfile.GFile(MODEL_SAVE_PATH, 'wb') as f:
  f.write(tflite_model)

##### **Step 3: Transfer Learning Approach**

Instead of creating our own CNN architecture, we can leverage an existing neural network such as the lightweight MobileNet image classification model.

Use the AI Assistant icon on the top of the cell to know more about MobileNet network by entering a prompt such as *'Tell me more about the MobileNet image classification model'*.

##### **Step 3(a):**

Repopulate the training folders for transfer learning. Notice the folder arrangement is the same as the LeNet custom model.  The image size is 224x224, which matches the size of the spectrograms.

In [0]:
import tensorflow as tf
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  ## hide INFO/WARNING logs

IMAGE_SIZE = 224; BATCH_SIZE = 5

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'esc_spec')
LABEL_FILE = os.path.join(BASE_DIR, 'sound_labels.txt')

datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.3)

train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    subset='training')

val_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    subset='validation')

In [0]:
image_batch, label_batch = next(val_generator)
image_batch.shape, label_batch.shape

print (train_generator.class_indices)

labels = '\n'.join(sorted(train_generator.class_indices.keys()))

with open(LABEL_FILE, 'w') as f:
  f.write(labels)

##### **Step 3(b):**

Identify a base model (such as MobileNetV2) and retain the layers of the feature detection part of the model. Do not include the classifier (top) part of the base model.

In [0]:
IMAGE_SIZE = 224 # The expected image shape is 224 x 224 pixels

IMG_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, 3)

# Create the base model from the pre-trained MobileNet V2

base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                              include_top=False,
                                              weights='imagenet')
base_model.trainable = False

##### **Step 3(c):**

Create a complete CNN model by combining the base model with a classifier.  The classifier output is a softmax layer and loss is set as categorical_crossentropy in compile method.

In [0]:
model_transfer = tf.keras.Sequential([
  base_model,
  tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'),
  tf.keras.layers.GlobalAveragePooling2D(),
  tf.keras.layers.Dense(units=11, activation='softmax')
])

print (model_transfer.summary())

In [0]:
model_transfer.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


##### **Step 3(d):**

Train the transfer learning model and plot training and validation accuracy.
Just like before, take a look at the CPU/GPU utilization while the model trains.

In [0]:
history = model_transfer.fit(train_generator,
                    steps_per_epoch=len(train_generator),
                    epochs=10,
                    validation_data=val_generator,
                    validation_steps= len(val_generator))

In [0]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
#plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),1])
plt.title('Validation Accuracy')
plt.show()

##### **Step 3(e): Compare Custom Model with Transfer Learning**


*   Validation accuracy on ESC-11 using Transfer Learning (MobileNetV2) is better.  
*   Accuracy is 75% with tranfer learning as compared to 60% with LeNet based CNN model.
*   The pretrained MobileNetV2 based model has about 2.6 million parameters as compared to LeNet style CNN which has 9.6 million parameters.

Save the new model as a TensorFlow Lite model.

In [0]:
# Convert the model to TFLite
MODEL_TL_SAVE_PATH = os.path.join(BASE_DIR, 'ESC-11-MobileNet.tflite')

converter = tf.lite.TFLiteConverter.from_keras_model(model_transfer)
tflite_model = converter.convert()

with tf.io.gfile.GFile(MODEL_TL_SAVE_PATH, 'wb') as f:
  f.write(tflite_model)


##### **Exercise 1:**

Working with a data centric approach.

*   Take a look at spectrogram images used to train ESC-11 model stored in directory (/content/esc_spec/). Think of ways to change the spectrogram images (or .wav files) to make it model friendly.
*   How would you put your idea in practice?

### Answer to Exercise 1

<details>
    <summary> Click here for our answer </summary>

    - Lots of Spectrograms have dark areas due to periods of silence in the 5s recording. If this silent period is eliminated and replaced by actual sound, it would help in creating a better (more accurate) model.
    - Preprocess the .wav file to exclude the periods of silence in the Spectrograms. The "silence" effect in SoX (Sound eXchange) is useful for detecting and manipulating periods of silence within audio files.
    
</details>



##### **Exercise 2:**

Use web based GUI based transfer learning for model creation.



*   Download the ESC-11 Zip file (containing .wav and spectrogram files) to your local computer.
*   Open [Teachable Machine](https://teachablemachine.withgoogle.com/) website and upload spectrogram images from different ESC-11 classes in your browser. Try to configure number of training epochs and batch size to see how it affects training. Take a look at the metrics (confusion matrix...) produced after training. The Teachable machine website uses transfer learning approach under the hood.





In [0]:
!wget https://edge-ai-doulos.s3.us-west-2.amazonaws.com/esc_spec.zip

##### Answer to Exercise 2

<details>
    <summary> Click here for the answer </summary>

    Request the instuctor to demonstrate the use of Teachable Machine for model training
    
</details>